# Re-execução FT-Entmax (bisseção corrigida) — Kaggle

**Este notebook agora roda SÓ o `FTTransformer_entmax`.** O SAINT foi retirado em 2026-09-18: a versão
que rodou aqui seguia as Equações 1–2 do artigo (**pós**-normalização), mas o código oficial
(`somepago/saint`, `RowColTransformer`) usa **pré**-norma, FFN GEGLU, `dim_head=64` na atenção de linha e
FF2 sobre a linha achatada. A pós-norma colapsa sob lr 1e-3 (TWS 5/10 sementes contra 0/10; ver
`results/saint_style_ablation.json`), o que produzia uma queda artificial de F1. O SAINT será refeito
separadamente com `style="reference"` (padrão desde o commit `d6667a6`); os registros de SAINT que já
existirem nos JSONs desta execução **não devem ser aproveitados**.

**Motivação do entmax:** a bisseção usava intervalo errado — a saída não era entmax-α. Corrigido em
`entmax_attention.py` (Jacobiana exata; testes de regressão). O F1 praticamente não muda
(Tier 1: 0,7447 → 0,7424), mas a atenção passa a ser genuinamente esparsa (zeros 0,015 → 0,080;
tokens efetivos 9,2 → 5,6), o que é o ponto: a neutralidade preditiva da esparsidade de atenção passa a
ser medida com um entmax correto.

**Fases** (cada uma grava um JSON próprio em `/kaggle/working`; nada sobrescreve os canônicos):
1. Tier 1 (10 datasets × 30 seeds, GridSearchCV) — **já concluído nesta sessão**
2. Tier 2 (6 datasets × 30 seeds, N=2000, GridSearchCV)
3. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 novo)
4. Ablação A — **os cinco Transformers sem o SAINT**, por transferência do Tier 1
   (`run_ablation_a_scaling.py`); corrige a assimetria descoberta na auditoria (a versão publicada
   re-tunou os Transformers em N=2000 enquanto os LSSVMs foram transferidos). O SAINT entra depois.
5. Ablações B + C (30 seeds)
6. Benchmark da Tabela 19 (`run_table19_benchmark.py`), só a variante do entmax

**Antes de rodar:** a célula 2 faz checkout de `revisao/estatistica-e-proveniencia` (commit `d6667a6` ou
posterior). Settings → Accelerator → **GPU T4 x2** (as células caras têm uma linha comentada com
`run_phase_2gpu`, que usa as duas placas com um processo cada e metade das sementes — ≈2× mais rápido e
metade da cota, já que a cota conta tempo de sessão; a Tabela 19 deve ficar em uma GPU só, por medir
tempo e VRAM). Os scripts são resumíveis: rodar de novo com o mesmo
`--output` continua de onde parou.

**Progresso:** cada fase imprime uma linha por execução concluída (`n/total`, tempo, ETA, último
resultado); o log completo fica em `/kaggle/working/<fase>.log`.

In [ ]:
# ── 1. GPU ──
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── 2. Repositório ──
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'   # branch com as correções (não é a main!)
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git branch --show-current
!git log --oneline -3
# Sanidade: o clone precisa conter as correções
import sys; sys.path.insert(0, '.')
from src.models.transformers.sparse_attention.entmax_attention import _EntmaxBisectFunction  # noqa
from src.models.ft_transformer_model import SAINTStage  # noqa
print('correções presentes: entmax (Function) + SAINTStage OK')

In [ ]:
# ── 3. Dependências e dados ──
# Todos os 16 datasets já vêm versionados em data/raw/*.parquet (inclusive AI4I): NÃO há download.
# (A versão anterior desta célula chamava scripts/download_data.py, que tenta o servidor da UCI e pode
#  ficar pendurada no Kaggle. Se isso aconteceu: Kernel → Interrupt, re-executar a célula 2 e esta.)
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
import sys, os; sys.path.insert(0, '.')
from pathlib import Path
faltando = [d for d in ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC','ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO']
            if not Path(f'data/raw/{d}.parquet').exists()]
assert not faltando, f'parquets ausentes no clone: {faltando} — o clone está na branch certa?'
from src.data.loaders import DatasetLoader
for ds in ['HAB','AI4I','TELCO']:      # amostra rápida: lê só do cache parquet
    X, y, _ = DatasetLoader.load(ds); print(f'{ds:<6} N={len(y):>5} p={X.shape[1]}')
print('dados OK (16 parquets presentes)')

In [ ]:
# ── 4. Configuração ──
import shutil, json
from pathlib import Path
MODELS = ['FTTransformer_entmax']          # SAINT retirado — ver cabeçalho
ABL_A_MODELS = ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax',
                'FTTransformer_sparsemax', 'FTTransformerCURColnorm']   # cinco, sem SAINT
MODELS_STR = ' '.join(MODELS)
ABL_A_STR  = ' '.join(ABL_A_MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2 = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
SEEDS30 = ' '.join(map(str, range(30)))
SEEDS20 = ' '.join(map(str, range(20)))
OUT = {
    'tier1':  'results/rerun_saint_entmax_tier1.json',
    'tier2':  'results/rerun_saint_entmax_tier2.json',
    'n5000':  'results/rerun_saint_entmax_n5000.json',
    'ablA':   'results/rerun_saint_entmax_ablA.json',
    'ablBC':  'results/rerun_saint_entmax_ablBC.json',
    'scal':   'results/rerun_saint_entmax_table19.json',
}
Path('results').mkdir(exist_ok=True)
# Resume: suba os JSONs parciais como Dataset do Kaggle e aponte o diretório
RESUME_DIR = None   # ex.: Path('/kaggle/input/rerun-saint-entmax')
if RESUME_DIR:
    for k, v in OUT.items():
        src = Path(RESUME_DIR) / Path(v).name
        if src.exists():
            shutil.copy(src, v); print('restaurado', v, len(json.load(open(v))))
def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)


# ── Execução com progresso enxuto ──
# Roda o script em segundo plano (log completo em /kaggle/working/<fase>.log) e imprime UMA linha
# sempre que o número de execuções concluídas no JSON de saída muda (ou a cada 15 min, como batimento).
import subprocess, time
nm = len(MODELS)
TOTAL = {'tier1': nm*10*30, 'tier2': nm*6*30, 'n5000': nm*6*30,
         'ablA': len(ABL_A_MODELS)*3*20, 'ablBC': nm*6*30, 'scal': 1*7}

def _count(path):
    try:
        r = json.load(open(path))
    except Exception:
        return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok:
        return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = f"último: {x.get('variant')}/{x.get('dataset', 'N=' + str(x.get('n')))}"
    info += f"/seed{x['seed']}" if 'seed' in x else ''
    info += f" F1={f1:.3f}" if isinstance(f1, (int, float)) else ''
    return len(ok), info

def run_phase(key, cmd, every=60, heartbeat=900):
    out, total = OUT[key], TOTAL[key]
    log = f'/kaggle/working/{key}.log'
    t0 = time.time(); last_n, last_print = -1, 0.0
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT)
        while True:
            rc = p.poll()
            n, info = _count(out)
            now = time.time()
            if n != last_n or rc is not None or now - last_print > heartbeat:
                el = (now - t0) / 60
                eta = el / n * (total - n) if n else float('nan')
                print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
                last_n, last_print = n, now
            if rc is not None:
                break
            time.sleep(every)
    print(f'[{key}] terminou (exit={rc}) — log: {log}')
    save(key)
    if rc != 0:
        raise RuntimeError(f'{key}: exit {rc} — veja {log} (tail abaixo)\n' + ''.join(open(log).readlines()[-15:]))


# ── Duas T4: um processo por GPU, sementes divididas ──────────────────────────
# Kaggle "GPU T4 x2" dá DUAS placas visíveis (cuda:0 e cuda:1). O PyTorch usa só a
# primeira se nada for feito. NÃO use DataParallel/DDP aqui: o SAINT e o FT-CUR fazem
# atenção INTER-INSTÂNCIAS dentro do lote, então dividir o lote entre as GPUs muda o
# modelo (a atenção passaria a ver metade das linhas), além de o ganho ser nulo em
# modelos deste tamanho. O jeito correto é paralelismo por experimento: dois processos
# independentes, cada um preso a uma GPU, cada um com METADE das sementes e seu PRÓPRIO
# arquivo de saída (dois processos gravando o mesmo JSON se sobrescrevem).
# Como a cota do Kaggle conta TEMPO DE SESSÃO e não GPU-hora, isto reduz a cota gasta
# quase pela metade.

def run_phase_2gpu(key, cmd_fmt, seeds=range(30), every=60, heartbeat=900):
    """cmd_fmt recebe {seeds} e {output} e é rodado duas vezes, uma por GPU."""
    import subprocess, time
    seeds = list(seeds)
    half = len(seeds) // 2
    partes = [(0, seeds[:half]), (1, seeds[half:])]
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs = [], [], []
    for gpu, sds in partes:
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        lf = open(log, 'w')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: {len(sds)} sementes -> {shard}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards)
        now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0) / 60; eta = el / n * (total - n) if n else float('nan')
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min', flush=True)
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs):
            break
        time.sleep(every)
    # junta os dois shards no arquivo canônico da fase
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros em {base}')
    save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs):
        raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')

import os
print('config OK — modelos:', MODELS)
for k, v in OUT.items():
    print(f'  {k:<6} -> {v}  (total {TOTAL[k]})')

In [ ]:
# ── 5. Tier 1 ──
# Uma GPU:
run_phase('tier1', f"python -u scripts/run_tier1_gridcv.py --models {MODELS_STR} --datasets {TIER1} --seeds {SEEDS30} --output {OUT['tier1']} --log-level WARNING")
# Duas T4 (≈2× mais rápido, metade da cota) — use ESTA no lugar da de cima:
# run_phase_2gpu('tier1', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR + " --datasets " + TIER1 + " --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 6. Tier 2 (N=2000) ──
run_phase('tier2', f"python -u scripts/run_tier2_gridcv.py --models {MODELS_STR} --datasets {TIER2} --seeds {SEEDS30} --n-train 2000 --output {OUT['tier2']} --log-level WARNING")
# Duas T4:
# run_phase_2gpu('tier2', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR + " --datasets " + TIER2 + " --n-train 2000 --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 7. Ablação D (N=5000, hiperparâmetros fixos = moda do Tier 2 NOVO) ──
# Não usar extract_tier2_fixed_params.py aqui: ele lê TODOS os results/tier2_transformers*.json
# (dumps antigos incluídos) e a deduplicação manteria os best_params da versão antiga.
import collections
recs = [r for r in json.load(open(OUT['tier2'])) if r.get('status') == 'ok' and r.get('best_params')]
by = collections.defaultdict(list)
for r in recs:
    by[(r['variant'], r['dataset'])].append(tuple(sorted(r['best_params'].items())))
cfg = json.load(open('config/tier2_fixed_params.json'))   # mantém as demais variantes
for (v, d), vals in sorted(by.items()):
    mode, cnt = collections.Counter(vals).most_common(1)[0]
    cfg.setdefault(v, {})[d] = dict(mode)
    print(f'{v:<22} {d:<9} {dict(mode)}  [moda {cnt}/{len(vals)}]')
Path('config/tier2_fixed_params_rerun.json').write_text(json.dumps(cfg, indent=2, sort_keys=True))
shutil.copy('config/tier2_fixed_params_rerun.json', '/kaggle/working/tier2_fixed_params_rerun.json')
run_phase('n5000', f"python -u scripts/run_tier2_fixedparams.py --models {MODELS_STR} --datasets {TIER2} --seeds {SEEDS30} --n-train 5000 --config config/tier2_fixed_params_rerun.json --output {OUT['n5000']} --log-level WARNING")

In [ ]:
# ── 8. Ablação A — CINCO Transformers (sem SAINT) por transferência do Tier 1 ──
# O SAINT será acrescentado depois, quando for refeito com style="reference"
# (run_ablation_a_scaling.py é resumível e chaveado por (variant, dataset, seed)).
!python scripts/merge_rerun_results.py --target results/tier1_gridcv.json --source {OUT['tier1']} --variants {MODELS_STR} --tag kaggle
run_phase('ablA', f"python -u scripts/run_ablation_a_scaling.py --models {ABL_A_STR} --tier1 results/tier1_gridcv.json --seeds {SEEDS20} --output {OUT['ablA']}")
# checagem: todos os registros devem ter protocol == transfer_from_tier1
import collections
recs = json.load(open(OUT['ablA']))
print(collections.Counter((r['variant'], r.get('protocol')) for r in recs if r.get('status') == 'ok'))

In [ ]:
# ── 9. Ablações B + C (30 seeds) ──
run_phase('ablBC', f"python -u scripts/run_tier1_gridcv.py --models {MODELS_STR} --datasets TWS_5f TWM_5f TWC_5f MKE MKM MKH --seeds {SEEDS30} --output {OUT['ablBC']} --log-level WARNING")
# Duas T4:
# run_phase_2gpu('ablBC', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR + " --datasets TWS_5f TWM_5f TWC_5f MKE MKM MKH --seeds {seeds} --output {output} --log-level WARNING")

In [ ]:
# ── 10. Benchmark de escalabilidade (Tabela 19) — só a variante do entmax ──
# As linhas de SAINT (mini e full-batch) serão refeitas junto com o SAINT.
run_phase('scal', f"python -u scripts/run_table19_benchmark.py --variants FTTransformer_entmax --repeats 3 --output {OUT['scal']}")
# NB: a Tabela 19 mede tempo e VRAM — rodar em UMA GPU e sem nada concorrente nela,
# senão as medições de memória e tempo ficam contaminadas.

In [ ]:
# ── 11. Resumo ──
import statistics as st
from collections import defaultdict
for key in ['tier1', 'tier2', 'n5000', 'ablA', 'ablBC']:
    p = Path(OUT[key])
    if not p.exists(): print(key, 'ausente'); continue
    ok = [r for r in json.load(open(p)) if r.get('status') == 'ok']
    f1 = defaultdict(list)
    for r in ok: f1[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
    print(f'\n=== {key}: {len(ok)} registros ok ===')
    for (v, d), vals in sorted(f1.items()):
        print(f'  {v:<22} {d:<9} {st.mean(vals):.4f} ± {st.pstdev(vals):.4f}  n={len(vals)}')